In [48]:
import pandas as pd
import numpy as np
import math
import warnings
warnings.filterwarnings('ignore')

In [26]:
df_placar2 = pd.read_csv("vlr_placar2.csv")
df_placar2.head(5)

,URL,Mapa,Time1,Mapas_T1,Score_T1,Time2,Mapas_T2,Score_T2,Vencedor
0,https://www.vlr.gg/596398/loud-vs-cloud9-vct-2...,Haven,LOUD,NaN,7,Cloud9,NaN,13,Cloud9
1,https://www.vlr.gg/596398/loud-vs-cloud9-vct-2...,Abyss,LOUD,NaN,11,Cloud9,NaN,13,Cloud9
2,https://www.vlr.gg/596398/loud-vs-cloud9-vct-2...,AllMaps,LOUD,0.0,18,Cloud9,2.0,26,Cloud9
3,https://www.vlr.gg/596399/envy-vs-evil-geniuse...,Abyss,ENVY,NaN,9,Evil Geniuses,NaN,13,Evil Geniuses
4,https://www.vlr.gg/596399/envy-vs-evil-geniuse...,Corrode,ENVY,NaN,13,Evil Geniuses,NaN,10,ENVY


In [25]:
df_stats2 = pd.read_csv("vlr_stats2.csv")
df_stats2.head(5)

,URL,Mapa_Num,Mapa,Jogador,Time,Agente,R,ACS,K,D,A,+/-,KAST,ADR,HS%,FK,FD,+/-_FK_FD
0,https://www.vlr.gg/596398/loud-vs-cloud9-vct-2...,0,Haven,Darker,LOUD,Viper,1.12,256,18,16,6,2,70,166,30,2,1,1
1,https://www.vlr.gg/596398/loud-vs-cloud9-vct-2...,0,Haven,cauanzin,LOUD,Sova,0.98,209,15,13,3,2,75,152,26,0,1,-1
2,https://www.vlr.gg/596398/loud-vs-cloud9-vct-2...,0,Haven,Virtyy,LOUD,Yoru,0.70,178,14,16,4,-2,55,102,35,2,3,-1
3,https://www.vlr.gg/596398/loud-vs-cloud9-vct-2...,0,Haven,pANcada,LOUD,Omen,0.67,145,11,13,1,-2,65,99,27,0,3,-3
4,https://www.vlr.gg/596398/loud-vs-cloud9-vct-2...,0,Haven,lukxo,LOUD,Killjoy,0.65,192,13,16,3,-3,60,125,35,4,4,0


# Juntas os DF's Placar e Stats

In [32]:
# Placar por mapa individual
df_placar_mapas = df_placar2[df_placar2['Mapa'] != 'AllMaps'].copy()

#  Placar AllMaps 
df_placar_all = df_placar2[df_placar2['Mapa'] == 'AllMaps'].copy()

# Filtra AllMaps do df de stats antes de qualquer merge
df_stats2 = df_stats2[df_stats2['Mapa'] != 'AllMaps'].copy()

#  Merge principal: stats + placar do mapa específico 
df_merged = df_stats2.merge(
    df_placar_mapas[['URL', 'Mapa', 'Time1', 'Time2', 'Score_T1', 'Score_T2', 'Vencedor']],
    on=['URL', 'Mapa'],
    how='left'
)

#  Merge secundário: dados do AllMaps 
df_merged = df_merged.merge(
    df_placar_all[['URL', 'Time1', 'Time2', 'Mapas_T1', 'Mapas_T2',
                   'Score_T1', 'Score_T2', 'Vencedor']],
    on='URL',
    how='left',
    suffixes=('', '_all')
)

#  Resolve colunas 
df_merged['Time1']    = df_merged['Time1'].fillna(df_merged['Time1_all'])
df_merged['Time2']    = df_merged['Time2'].fillna(df_merged['Time2_all'])
df_merged['Vencedor'] = df_merged['Vencedor'].fillna(df_merged['Vencedor_all'])

df_merged['Score_T1'] = df_merged['Score_T1'].fillna(df_merged['Score_T1_all'])
df_merged['Score_T2'] = df_merged['Score_T2'].fillna(df_merged['Score_T2_all'])

#  Adversário 
df_merged['Adversario'] = df_merged.apply(
    lambda row: row['Time2'] if row['Time'] == row['Time1'] else row['Time1'],
    axis=1
)

#  Scores 
df_merged['Score_Time'] = df_merged.apply(
    lambda row: row['Score_T1'] if row['Time'] == row['Time1'] else row['Score_T2'],
    axis=1
)

df_merged['Score_Adv'] = df_merged.apply(
    lambda row: row['Score_T2'] if row['Time'] == row['Time1'] else row['Score_T1'],
    axis=1
)

#  Mapas ganhos 
df_merged['Mapas_Time'] = df_merged.apply(
    lambda row: row['Mapas_T1'] if row['Time'] == row['Time1'] else row['Mapas_T2'],
    axis=1
)

df_merged['Mapas_Adv'] = df_merged.apply(
    lambda row: row['Mapas_T2'] if row['Time'] == row['Time1'] else row['Mapas_T1'],
    axis=1
)

#  Remove auxiliares 
df_merged = df_merged.drop(columns=[
    'Time1', 'Time2', 'Time1_all', 'Time2_all',
    'Score_T1', 'Score_T2',
    'Score_T1_all', 'Score_T2_all',
    'Mapas_T1', 'Mapas_T2',
    'Vencedor_all'
])

#  Mapeamento correto 
nome_para_tag = {
    'Sentinels': 'SEN',
    'NRG': 'NRG',
    'Cloud9': 'C9',
    'FURIA': 'FUR',
    'LEVIATÁN': 'LEV',
    'Evil Geniuses': 'EG',
    '100 Thieves': '100T',
    'KRÜ Esports': 'KRU',
    'VISA KRÜ(KRÜ Esports)': 'KRU',
    'G2 Esports': 'G2',
    'ENVY': 'ENVY',
    'MIBR': 'MIBR',
    'LOUD': 'LOUD',
    '2Game Esports': '2G'
}

for col in ['Time', 'Vencedor', 'Adversario']:
    df_merged[col] = df_merged[col].map(nome_para_tag).fillna(df_merged[col])

df_merged.to_csv("merged_status_placar.csv", index=False)

In [33]:
df_merged.head(5)

,URL,Mapa_Num,Mapa,Jogador,Time,Agente,R,ACS,K,D,...,HS%,FK,FD,+/-_FK_FD,Vencedor,Adversario,Score_Time,Score_Adv,Mapas_Time,Mapas_Adv
0,https://www.vlr.gg/596398/loud-vs-cloud9-vct-2...,0,Haven,Darker,LOUD,Viper,1.12,256,18,16,...,30,2,1,1,C9,C9,7,13,0.0,2.0
1,https://www.vlr.gg/596398/loud-vs-cloud9-vct-2...,0,Haven,cauanzin,LOUD,Sova,0.98,209,15,13,...,26,0,1,-1,C9,C9,7,13,0.0,2.0
2,https://www.vlr.gg/596398/loud-vs-cloud9-vct-2...,0,Haven,Virtyy,LOUD,Yoru,0.70,178,14,16,...,35,2,3,-1,C9,C9,7,13,0.0,2.0
3,https://www.vlr.gg/596398/loud-vs-cloud9-vct-2...,0,Haven,pANcada,LOUD,Omen,0.67,145,11,13,...,27,0,3,-3,C9,C9,7,13,0.0,2.0
4,https://www.vlr.gg/596398/loud-vs-cloud9-vct-2...,0,Haven,lukxo,LOUD,Killjoy,0.65,192,13,16,...,35,4,4,0,C9,C9,7,13,0.0,2.0


# Descobrir as Features mais importantes para um time ganhar

In [34]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

df_merged = pd.read_csv("merged_status_placar.csv")
df = df_merged.copy()
df['Win'] = (df['Time'] == df['Vencedor']).astype(int)
cols_drop = [
    'Mapa',
    'Time',
    'Agente',
    'Vencedor',
    'Adversario',
    'Score_Time',
    'Score_Adv',
    'Mapas_Time',
    'Mapas_Adv',
    'URL',
    'Jogador'
]

df_model = df.drop(columns=cols_drop)

X = df_model.drop(columns=['Win'])
y = df_model['Win']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model = RandomForestClassifier(n_estimators=200, random_state=42)
model.fit(X_train, y_train)

importances = model.feature_importances_

feat_importance = pd.Series(importances, index=X.columns)
feat_importance = feat_importance.sort_values(ascending=False)

print(feat_importance.head(20))

+/-          0.142106
KAST         0.131555
D            0.115082
R            0.099738
ACS          0.092948
ADR          0.081084
HS%          0.071841
A            0.065128
K            0.061822
+/-_FK_FD    0.041530
FK           0.037149
FD           0.035556
Mapa_Num     0.024461
dtype: float64


# Calculo de Impacto

In [40]:
REF_MEDIANO = {
    'R':          0.98,
    'ACS':        192.0,
    'K':          14.0,
    'D':          15.0,
    'A':          5.0,
    '+/-':        0.0,
    'KAST':       73.0,
    'ADR':        126.0,
    'HS':        27.0,
    'FK':         2.0,
    'FD':         2.0,
    '+/-_FK_FD':  0.0,
    'Mapa_Num':   2.0,
    'Score_Time': 12.5,
    'Score_Adv':  13.0,
    'Mapas_Time': 1.0,
    'Mapas_Adv':  2.0
}

PESOS = {
    '+/-':        0.142,
    'KAST':       0.132,
    'D':          0.115,
    'R':          0.100,
    'ACS':        0.093,
    'ADR':        0.081,
    'HS':        0.072,
    'A':          0.065,
    'K':          0.062,
    '+/-_FK_FD':  0.042,
    'FK':         0.037,
    'FD':         0.036,
    'Mapa_Num':   0.024
}


def calcular_impacto(r, acs, kast, adr, hs, tot_k, tot_d, tot_fk, tot_fd, jogos):
    def clip(x, min_v=0.5, max_v=1.5):
        return max(min(x, max_v), min_v)

    kd_rate  = (tot_k - tot_d) / jogos if jogos > 0 else 0
    fkd_rate = (tot_fk - tot_fd) / jogos if jogos > 0 else 0

    scores = {
        'R':          clip(r    / REF_MEDIANO['R']),
        'ACS':        clip(acs  / REF_MEDIANO['ACS']),
        'ADR':        clip(adr  / REF_MEDIANO['ADR']),
        'KAST':       clip(kast / REF_MEDIANO['KAST']),
        'HS':         clip(math.sqrt(hs / REF_MEDIANO['HS'])),
        '+/-':        clip(1 + (kd_rate  / 2)),  
        '+/-_FK_FD':  clip(1 + (fkd_rate / 1)),   
    }

    score = sum(PESOS[m] * scores[m] for m in scores)
    return round(score, 3)


# DF do jogador com o status e seu impacto por mapa

In [54]:
def time_atual():
    df = pd.read_csv("vlr_stats_2026.csv")
    
    tag = df_stats2["Time"].unique()
    
    times = {}
    
    for sigla in tag:
        jogadores = df[df["Time"] == sigla]["Jogador"].unique()
        times[sigla] = list(jogadores)
    
    return times


times = time_atual()

times


jogador_time_atual = {
    jogador: sigla
    for sigla, jogadores in times.items()
    for jogador in jogadores
}

mapas = df_merged["Mapa"].unique()

resultados = []

for mapa in mapas:
    df_mapa = df_merged[df_merged["Mapa"] == mapa].copy()

    if df_mapa.empty:
        print(f"{mapa} — sem dados")
        continue

    for jogador, df_j in df_mapa.groupby("Jogador"):

        tot_k  = int(df_j["K"].sum())
        tot_d  = int(df_j["D"].sum())
        tot_a  = int(df_j["A"].sum())
        tot_fk = int(df_j["FK"].sum())
        tot_fd = int(df_j["FD"].sum())

        med_r    = df_j["R"].mean()
        med_acs  = df_j["ACS"].mean()
        med_kast = df_j["KAST"].mean()
        med_adr  = df_j["ADR"].mean()
        med_hs   = df_j["HS%"].mean()

        jogos   = len(df_j)
        wins    = (df_j["Time"] == df_j["Vencedor"]).sum()
        winrate = wins / jogos if jogos > 0 else 0

        time_atual_jogador = (
            jogador_time_atual.get(jogador)
            or df_j["Time"].mode()[0]  
        )

        impacto = calcular_impacto(
            r=med_r, acs=med_acs, kast=med_kast, adr=med_adr, hs=med_hs,
            tot_k=tot_k, tot_d=tot_d, tot_fk=tot_fk, tot_fd=tot_fd,
            jogos=jogos
        )

        resultados.append({
            "Jogador":   jogador,
            "Time":      time_atual_jogador,
            "Mapa":      mapa,
            "R":         round(med_r,    3),
            "ACS":       round(med_acs,  1),
            "K":         tot_k,
            "D":         tot_d,
            "A":         tot_a,
            "+/-":       tot_k - tot_d,
            "KAST":      round(med_kast, 1),
            "ADR":       round(med_adr,  1),
            "HS%":       round(med_hs,   1),
            "FK":        tot_fk,
            "FD":        tot_fd,
            "+/-_FK_FD": tot_fk - tot_fd,
            "Jogos":     jogos,
            "Wins":      int(wins),
            "WinRate":   round(winrate, 2),
            "Impacto":   impacto,
        })

    print(f"✅ {mapa:<12} — processado")


df_final = (
    pd.DataFrame(resultados)
    .sort_values(["Mapa", "Impacto"], ascending=[True, False])
    .reset_index(drop=True)
)

print(f"\n📦 {df_final['Mapa'].nunique()} mapas | {len(df_final)} linhas")

df_final.to_csv("df_jogador_por_mapas.csv", index=False)

df_final

✅ Haven        — processado
✅ Abyss        — processado
✅ Bind         — processado
✅ Corrode      — processado
✅ Split        — processado
✅ Pearl        — processado
✅ Breeze       — processado
✅ Lotus        — processado
✅ Icebox       — processado
✅ Sunset       — processado
✅ Ascent       — processado
✅ Fracture     — processado

📦 12 mapas | 887 linhas


,Jogador,Time,Mapa,R,ACS,K,D,A,+/-,KAST,ADR,HS%,FK,FD,+/-_FK_FD,Jogos,Wins,WinRate,Impacto
0,infiltrator,KRÜ,Abyss,1.360,293.5,43,30,6,13,87.5,186.0,32.0,5,2,3,2,0,0.0,0.841
1,lukxo,LOUD,Abyss,1.443,279.7,77,44,8,33,75.7,174.7,34.7,12,10,2,3,0,0.0,0.816
2,koalanoob,FUR,Abyss,1.250,271.3,58,41,18,17,79.3,169.0,28.3,13,9,4,3,3,1.0,0.796
3,vora,100T,Abyss,1.385,255.5,33,22,6,11,82.0,163.5,28.5,2,2,0,2,1,0.5,0.782
4,alym,FUR,Abyss,1.330,239.7,51,34,22,17,81.0,147.7,27.3,12,4,8,3,3,1.0,0.778
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
882,RobbieBk,LOUD,Sunset,0.810,117.5,18,24,5,-6,65.0,82.5,44.5,4,5,-1,2,1,0.5,0.408
883,s0m,NRG,Sunset,0.640,152.0,44,68,19,-24,55.8,104.0,36.5,6,16,-10,4,0,0.0,0.404
884,tuyz,FUR,Sunset,0.565,147.0,16,30,13,-14,64.0,92.0,34.0,2,6,-4,2,0,0.0,0.401
885,Urango,FUR,Sunset,0.525,119.0,13,29,8,-16,63.5,88.0,37.5,0,4,-4,2,0,0.0,0.380


In [60]:
df_final[df_final["Jogador"] == 'zekken']

,Jogador,Time,Mapa,R,ACS,K,D,A,+/-,KAST,ADR,HS%,FK,FD,+/-_FK_FD,Jogos,Wins,WinRate,Impacto
50,zekken,MIBR,Abyss,0.906,182.6,85,92,39,-7,69.0,125.0,28.7,20,22,-2,7,3,0.43,0.494
96,zekken,MIBR,Ascent,1.012,218.2,70,65,13,5,72.2,139.2,27.8,20,20,0,4,2,0.50,0.691
162,zekken,MIBR,Bind,1.047,200.5,63,55,20,8,75.5,130.5,21.2,13,11,2,4,3,0.75,0.708
241,zekken,MIBR,Breeze,1.297,237.3,46,36,16,10,80.7,150.3,27.0,11,6,5,3,2,0.67,0.774
297,zekken,MIBR,Corrode,1.226,247.3,126,101,40,25,77.4,159.6,29.9,26,21,5,7,5,0.71,0.772
412,zekken,MIBR,Fracture,0.880,160.0,10,13,2,-3,61.0,113.0,25.0,2,3,-1,1,0,0.00,0.448
454,zekken,MIBR,Haven,1.031,205.5,153,143,54,10,73.5,129.1,28.1,29,37,-8,11,5,0.45,0.655
525,zekken,MIBR,Icebox,0.970,212.8,84,78,23,6,70.4,138.6,33.8,20,13,7,5,2,0.40,0.702
592,zekken,MIBR,Lotus,1.127,241.1,192,156,47,36,74.2,153.8,21.5,45,46,-1,11,7,0.64,0.724
669,zekken,MIBR,Pearl,1.125,243.0,195,171,59,24,72.3,154.0,26.4,47,37,10,10,7,0.70,0.746


# DF do jogador com o status e seu impacto geral

In [61]:
times = {
    'LOUD': ['Darker', 'cauanzin', 'Virtyy', 'pANcada', 'lukxo'],
    'C9':   ['OXY', 'penny', 'v1c', 'Xeppaa', 'Zellsis', 'Demon1'],
    'G2':   ['BABYBAY', 'trent', 'valyn', 'leaf', 'jawgemo'],
    '100T': ['Cryocells', 'bang', 'vora', 'Asuna', 'Timotino'],
    'ENVY': ['Rossy', 'Inspire', 'keznit', 'Eggsterr', 'P0PPIN', 'Demon1'],
    'EG':   ['dgzin', 'bao', 'Okeanos', 'C0M', 'supamen'],
    'KRÜ':  ['benG', 'mta', 'NagZ', 'infiltrator', 'Governor'],
    'FUR':  ['koalanoob', 'artzin', 'nerve', 'eeiu', 'alym'],
    'LEV':  ['PxS', 'Sato', 'kiNgg', 'blowz', 'spike', 'Neon'],
    'NRG':  ['brawk', 'keiko', 'mada', 'Ethan', 'skuba'],
    'MIBR': ['aspas', 'Mazino', 'Verno', 'tex', 'zekken'],
    'SEN':  ['johnqt', 'cortezia', 'Reduxx', 'Kyu', 'N4RRATE'],
}

# função para achar o time
def pegar_time(jogador, df_j):
    for time, players in times.items():
        if jogador in players:
            return time
    return df_j.sort_values("Mapa_Num")["Time"].iloc[-1]


linhas = []

for jogador in df_merged["Jogador"].unique():
    df_j = df_merged[df_merged["Jogador"] == jogador]

    # SOMAS 
    tot_k  = int(df_j["K"].sum())
    tot_d  = int(df_j["D"].sum())
    tot_a  = int(df_j["A"].sum())
    tot_fk = int(df_j["FK"].sum())
    tot_fd = int(df_j["FD"].sum())

    score_time = df_j["Score_Time"].sum()
    score_adv  = df_j["Score_Adv"].sum()
    mapas_time = df_j["Mapas_Time"].sum()
    mapas_adv  = df_j["Mapas_Adv"].sum()

    # MÉDIAS
    med_r    = df_j["R"].mean()
    med_acs  = df_j["ACS"].mean()
    med_kast = df_j["KAST"].mean()
    med_adr  = df_j["ADR"].mean()
    med_hs   = df_j["HS%"].mean()

    # DERIVADOS 
    diff_kd = tot_k - tot_d
    diff_fk_fd = tot_fk - tot_fd

    # TIME
    time_final = pegar_time(jogador, df_j)

    linhas.append({
        "Jogador": jogador,
        "Time": time_final,

        "K": tot_k,
        "D": tot_d,
        "A": tot_a,
        "+/-": diff_kd,

        "FK": tot_fk,
        "FD": tot_fd,
        "+/-_FK_FD": diff_fk_fd,

        "R": med_r,
        "ACS": med_acs,
        "KAST": med_kast,
        "ADR": med_adr,
        "HS%": med_hs,

        "Score_Time": score_time,
        "Score_Adv": score_adv,
        "Mapas_Time": mapas_time,
        "Mapas_Adv": mapas_adv,
    })

df_geral = pd.DataFrame(linhas)

# Formatação
df_geral["R"]    = df_geral["R"].round(3)
df_geral["ACS"]  = df_geral["ACS"].round(1)
df_geral["KAST"] = df_geral["KAST"].round(1)
df_geral["ADR"]  = df_geral["ADR"].round(1)
df_geral["HS%"]  = df_geral["HS%"].round(1)

# Inteiros
for col in ["K", "D", "A", "+/-", "FK", "FD", "+/-_FK_FD",
            "Score_Time", "Score_Adv", "Mapas_Time", "Mapas_Adv"]:
    df_geral[col] = pd.to_numeric(df_geral[col], errors="coerce").fillna(0).astype(int)

# Ordem das colunas
colunas = [
    "Jogador", "Time",
    "R", "ACS", "KAST", "ADR", "HS%",
    "K", "D", "A", "+/-",
    "FK", "FD", "+/-_FK_FD",
    "Score_Time", "Score_Adv",
    "Mapas_Time", "Mapas_Adv"
]

df_geral

,Jogador,Time,K,D,A,+/-,FK,FD,+/-_FK_FD,R,ACS,KAST,ADR,HS%,Score_Time,Score_Adv,Mapas_Time,Mapas_Adv
0,Darker,LOUD,131,144,56,-13,17,17,0,0.919,193.3,69.8,122.3,24.7,74,116,3,18
1,cauanzin,LOUD,624,712,256,-88,74,104,-30,0.894,187.4,68.6,124.4,25.0,446,531,38,79
2,Virtyy,LOUD,324,347,80,-23,72,91,-19,0.892,196.8,66.4,122.2,24.8,200,266,15,41
3,pANcada,LOUD,632,656,321,-24,66,54,12,0.992,183.2,72.5,125.0,36.0,446,531,38,79
4,lukxo,LOUD,565,475,122,90,102,73,29,1.105,234.5,68.5,152.5,32.6,296,373,24,59
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
96,Add3r,FUR,28,28,3,0,4,6,-2,0.870,194.5,65.5,125.0,32.5,26,12,4,0
97,stellar,FUR,29,24,11,5,4,6,-2,1.075,204.5,76.5,143.5,19.0,26,12,4,0
98,jakee,FUR,29,25,18,4,1,4,-3,1.180,221.0,82.0,143.5,25.0,26,12,4,0
99,mwzera,FUR,44,55,23,-11,7,7,0,0.872,166.2,66.0,111.0,19.5,40,40,4,4


In [44]:
df_geral.describe()

,K,D,A,+/-,FK,FD,+/-_FK_FD,R,ACS,KAST,ADR,HS%,Score_Time,Score_Adv,Mapas_Time,Mapas_Adv
count,101.000000,101.000000,101.000000,101.000000,101.000000,101.000000,101.000000,101.000000,101.000000,101.000000,101.000000,101.000000,101.000000,101.000000,101.000000,101.000000
mean,496.910891,497.574257,206.900990,-0.663366,72.247525,72.247525,0.000000,0.958990,190.644554,70.726733,125.033663,28.102970,358.366337,364.504950,44.405941,47.970297
std,364.597809,329.235094,168.257927,76.882804,66.394037,57.760264,19.038382,0.127589,22.683040,4.380637,13.844351,4.829316,243.441891,252.712905,32.828396,36.972816
min,28.000000,21.000000,3.000000,-138.000000,1.000000,0.000000,-55.000000,0.573000,136.100000,58.900000,91.000000,19.000000,26.000000,12.000000,3.000000,0.000000
25%,180.000000,194.000000,69.000000,-44.000000,24.000000,27.000000,-10.000000,0.881000,177.800000,68.300000,115.700000,24.800000,138.000000,145.000000,16.000000,19.000000
50%,385.000000,420.000000,147.000000,-13.000000,60.000000,62.000000,-2.000000,0.955000,191.300000,71.100000,125.000000,27.600000,274.000000,315.000000,34.000000,40.000000
75%,754.000000,741.000000,324.000000,23.000000,98.000000,100.000000,7.000000,1.058000,202.700000,73.900000,132.700000,31.500000,561.000000,554.000000,67.000000,77.000000
max,1307.000000,1205.000000,664.000000,264.000000,312.000000,265.000000,75.000000,1.275000,239.100000,82.000000,152.500000,40.500000,882.000000,890.000000,125.000000,133.000000


In [62]:
REF_MEDIANO = {
    'K':          385.0,
    'D':          420.0,
    'A':          147.0,
    '+/-':        -13.0,
    'FK':         60.0,
    'FD':         62.0,
    '+/-_FK_FD':  -2.0,
    'R':          0.955,
    'ACS':        191.3,
    'KAST':       71.1,
    'ADR':        125.0,
    'HS':        27.6,
    'Score_Time': 274.0,
    'Score_Adv':  315.0,
    'Mapas_Time': 34.0,
    'Mapas_Adv':  40.0
}

PESOS = {
    '+/-':        0.142,
    'KAST':       0.132,
    'D':          0.115,
    'R':          0.100,
    'ACS':        0.093,
    'ADR':        0.081,
    'HS%':        0.072,
    'A':          0.065,
    'K':          0.062,
    '+/-_FK_FD':  0.042,
    'FK':         0.037,
    'FD':         0.036,
    'Mapa_Num':   0.024
}

def calcular_impacto(r, acs, kast, adr, hs, tot_k, tot_d, tot_fk, tot_fd, jogos):
    def clip(x, min_v=0.5, max_v=1.5):
        return max(min(x, max_v), min_v)

    kd_rate  = (tot_k - tot_d) / jogos if jogos > 0 else 0
    fkd_rate = (tot_fk - tot_fd) / jogos if jogos > 0 else 0

    scores = {
        'R':         clip(r    / REF_MEDIANO['R']),
        'ACS':       clip(acs  / REF_MEDIANO['ACS']),
        'ADR':       clip(adr  / REF_MEDIANO['ADR']),
        'KAST':      clip(kast / REF_MEDIANO['KAST']),
        'HS':        clip(math.sqrt(hs / REF_MEDIANO['HS'])),
        '+/-':       clip(1 + (kd_rate  / 2)),
        '+/-_FK_FD': clip(1 + (fkd_rate / 1)),
    }

    pesos_usados = {k: v for k, v in PESOS.items() if k in scores}
    score = sum(pesos_usados[m] * scores[m] for m in pesos_usados)
    return round(score, 3)

df_geral['jogos'] = df_geral['Mapas_Time'] + df_geral['Mapas_Adv']

df_geral['Impacto'] = df_geral.apply(lambda row: calcular_impacto(
    r=row['R'],
    acs=row['ACS'],
    kast=row['KAST'],
    adr=row['ADR'],
    hs=row['HS%'],
    tot_k=row['K'],
    tot_d=row['D'],
    tot_fk=row['FK'],
    tot_fd=row['FD'],
    jogos=row['jogos']
), axis=1)

df_geral.drop(columns='jogos', inplace=True)
df_geral.to_csv("df_jogador_geral.csv", index=False)
df_geral

,Jogador,Time,K,D,A,+/-,FK,FD,+/-_FK_FD,R,ACS,KAST,ADR,HS%,Score_Time,Score_Adv,Mapas_Time,Mapas_Adv,Impacto
0,Darker,LOUD,131,144,56,-13,17,17,0,0.919,193.3,69.8,122.3,24.7,74,116,3,18,0.539
1,cauanzin,LOUD,624,712,256,-88,74,104,-30,0.894,187.4,68.6,124.4,25.0,446,531,38,79,0.513
2,Virtyy,LOUD,324,347,80,-23,72,91,-19,0.892,196.8,66.4,122.2,24.8,200,266,15,41,0.532
3,pANcada,LOUD,632,656,321,-24,66,54,12,0.992,183.2,72.5,125.0,36.0,446,531,38,79,0.582
4,lukxo,LOUD,565,475,122,90,102,73,29,1.105,234.5,68.5,152.5,32.6,296,373,24,59,0.725
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
96,Add3r,FUR,28,28,3,0,4,6,-2,0.870,194.5,65.5,125.0,32.5,26,12,4,0,0.551
97,stellar,FUR,29,24,11,5,4,6,-2,1.075,204.5,76.5,143.5,19.0,26,12,4,0,0.681
98,jakee,FUR,29,25,18,4,1,4,-3,1.180,221.0,82.0,143.5,25.0,26,12,4,0,0.710
99,mwzera,FUR,44,55,23,-11,7,7,0,0.872,166.2,66.0,111.0,19.5,40,40,4,4,0.480


# DF do jogador com o status e seu impacto por agente

In [64]:
agentes = df_merged["Agente"].unique()

resultados_agente = []
for agente in agentes:
    df_agente = df_merged[df_merged["Agente"] == agente].copy()
    if df_agente.empty:
        continue
    for jogador, df_j in df_agente.groupby("Jogador"):
        tot_k  = int(df_j["K"].sum())
        tot_d  = int(df_j["D"].sum())
        tot_a  = int(df_j["A"].sum())
        tot_fk = int(df_j["FK"].sum())
        tot_fd = int(df_j["FD"].sum())
        med_r    = df_j["R"].mean()
        med_acs  = df_j["ACS"].mean()
        med_kast = df_j["KAST"].mean()
        med_adr  = df_j["ADR"].mean()
        med_hs   = df_j["HS%"].mean()
        jogos   = len(df_j)
        wins    = (df_j["Time"] == df_j["Vencedor"]).sum()
        winrate = wins / jogos if jogos > 0 else 0
        time_atual_jogador = (
            jogador_time_atual.get(jogador)
            or df_j["Time"].mode()[0]
        )
        impacto = calcular_impacto(
            r=med_r, acs=med_acs, kast=med_kast, adr=med_adr, hs=med_hs,
            tot_k=tot_k, tot_d=tot_d, tot_fk=tot_fk, tot_fd=tot_fd,
            jogos=jogos
        )
        resultados_agente.append({
            "Jogador":   jogador,
            "Time":      time_atual_jogador,
            "Agente":    agente,
            "R":         round(med_r,    3),
            "ACS":       round(med_acs,  1),
            "K":         tot_k,
            "D":         tot_d,
            "A":         tot_a,
            "+/-":       tot_k - tot_d,
            "KAST":      round(med_kast, 1),
            "ADR":       round(med_adr,  1),
            "HS%":       round(med_hs,   1),
            "FK":        tot_fk,
            "FD":        tot_fd,
            "+/-_FK_FD": tot_fk - tot_fd,
            "Jogos":     jogos,
            "Wins":      int(wins),
            "WinRate":   round(winrate, 2),
            "Impacto":   impacto,
        })
    print(f"✅ {agente:<14} — processado")

df_final_agente = (
    pd.DataFrame(resultados_agente)
    .sort_values(["Agente", "Impacto"], ascending=[True, False])
    .reset_index(drop=True)
)
print(f"\n📦 {df_final_agente['Agente'].nunique()} agentes | {len(df_final_agente)} linhas")
df_final_agente.to_csv("df_jog_agente.csv", index=False)
df_final_agente

✅ Viper          — processado
✅ Sova           — processado
✅ Yoru           — processado
✅ Omen           — processado
✅ Killjoy        — processado
✅ Neon           — processado
✅ Chamber        — processado
✅ Astra          — processado
✅ Harbor         — processado
✅ Veto           — processado
✅ Breach         — processado
✅ Waylay         — processado
✅ Skye           — processado
✅ Brimstone      — processado
✅ Fade           — processado
✅ Tejo           — processado
✅ Sage           — processado
✅ Iso            — processado
✅ Kayo           — processado
✅ Jett           — processado
✅ Cypher         — processado
✅ Raze           — processado
✅ Phoenix        — processado
✅ Vyse           — processado
✅ Deadlock       — processado
✅ Gekko          — processado
✅ Clove          — processado
✅ Reyna          — processado

📦 28 agentes | 558 linhas


,Jogador,Time,Agente,R,ACS,K,D,A,+/-,KAST,ADR,HS%,FK,FD,+/-_FK_FD,Jogos,Wins,WinRate,Impacto
0,johnqt,SEN,Astra,1.500,234.0,47,21,14,26,92.0,149.5,32.0,2,0,2,2,1,0.50,0.807
1,s0m,NRG,Astra,1.433,237.7,49,31,31,18,88.0,151.0,23.7,8,1,7,3,2,0.67,0.803
2,Melser,KRÜ,Astra,1.248,216.6,77,63,37,14,77.2,144.4,35.6,13,6,7,5,0,0.00,0.749
3,Demon1,ENVY,Astra,1.175,232.0,42,30,12,12,84.5,149.5,41.0,4,4,0,2,1,0.50,0.745
4,Rossy,ENVY,Astra,1.270,202.0,39,32,30,7,75.5,133.5,19.5,3,2,1,2,1,0.50,0.734
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
553,artzin,FUR,Yoru,0.823,185.6,102,118,32,-16,67.0,119.6,22.3,25,33,-8,7,2,0.29,0.470
554,aspas,MIBR,Yoru,0.720,150.0,10,15,3,-5,64.0,103.0,25.0,4,3,1,1,0,0.00,0.468
555,Virtyy,LOUD,Yoru,0.720,171.8,74,97,24,-23,60.7,106.2,24.8,15,18,-3,6,0,0.00,0.432
556,spike,LEV,Yoru,0.690,186.0,32,43,6,-11,48.3,120.0,41.7,7,12,-5,3,0,0.00,0.422
